<div style="padding:20px; border:1px solid #d0d7de; border-radius:12px; background:#f8fafc;">

<div align="center">

# CMAscan: Detection and Scoring of CMA-Targeting Motifs

</div>

CMAscan scans multi-FASTA protein sequences and reports canonical, phospho-dependent, and acetyl-dependent CMA motif candidates with integrated **cPSSM**, **ePSSM**, and **IUPred3** scoring.

**Runtime requirements**
- Internet access is required to download scoring resources.
- Large input files (hundreds of proteins) may take 10–30 minutes. Do not
   close the browser tab while the pipeline is running.
- Output column `localization` is reported as **1-based motif start position** in the input protein sequence.
- PTM-site likelihood prediction (MusiteDeep) is planned for v1.1.

**Quick Start**
1. Click **Runtime → Run all** to start the pipeline.
2. After a few seconds, **Step 1** will display a **"Choose Files"** button — use it to upload your `.fasta`, `.faa`, or `.txt` file. The pipeline pauses here until a file is selected.
3. The pipeline then continues automatically. Pseudo-PTM options can be adjusted in **Step 1** form fields before clicking Run all (defaults are recommended).
4. The output CSV is previewed in **Step 12** and downloaded automatically.

*Alternative input mode: if your file is already inside the Colab runtime
(e.g. uploaded to `/content/` via the Files panel, or mounted from Google Drive),
set `INPUT_MODE = 'path'` in Step 1 and provide the in-Colab path.
Paths from your local computer will not work — Colab cannot access your local filesystem.*

If auto-download does not start, rerun **Step 12** or download the file manually from `/content` in the left **Files** panel.

**Troubleshooting**
If the pipeline stops, shows an error, or the "Choose Files" button does not
appear:
1. Click **Runtime → Disconnect and delete runtime**.
2. Click **Runtime → Run all** to restart from scratch.

Do not run individual cells manually — the pipeline requires all previous
steps to be executed in order.

**External tools**
This notebook uses IUPred3 (https://iupred3.elte.hu/) for disorder scoring.
If you publish results obtained with CMAscan, please also cite the IUPred3
authors — see **Step 8** for full references.

</div>


In [ ]:
#@title Step 1: Upload or Select Input File
# Local import keeps this cell runnable as the first step in Colab.
from google.colab import files

# Input mode settings.
INPUT_MODE = 'upload'  #@param ['upload', 'path']
INPUT_FILE_PATH = '/content/your_file.fasta'  #@param {type:'string'}
OUTPUT_FILE_PATH = '/content/motif_hits_scored.csv'  #@param {type:'string'}

# Pseudo-PTM settings (user-friendly switches).
USE_PSEUDOACETYLATION = True  #@param {type:'boolean'}
USE_PSEUDOPHOSPHORYLATION = True  #@param {type:'boolean'}
# PSEUDOPHOSPHO_RESIDUE: residue used to mimic phosphorylated S/T/Y in PSSM
# scoring. Default 'E' is recommended for in silico screening; switch to 'D'
# only if you have a specific reason. See manuscript for rationale.
PSEUDOPHOSPHO_RESIDUE = 'E'  #@param ['E', 'D']

if INPUT_MODE == 'upload':
    uploaded = files.upload()
    if not uploaded:
        raise ValueError('No file uploaded. Please upload one .fasta/.faa/.txt file.')
    INPUT_FILE_PATH = next(iter(uploaded.keys()))
    print(f'Selected uploaded file: {INPUT_FILE_PATH}')
elif INPUT_MODE == 'path':
    print(f'Using manual path: {INPUT_FILE_PATH}')
else:
    raise ValueError("INPUT_MODE must be 'upload' or 'path'.")

This step defines the input source and user-facing options used by the full pipeline.


In [ ]:
#@title Step 2: Imports and Global Settings
import csv
import json
import pickle
import subprocess
import tarfile
import tempfile
import time
import urllib.request
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, Iterator, List, Sequence
from urllib.parse import quote

from google.colab import files
import pandas as pd

ALLOWED_EXTENSIONS = {'.fasta', '.faa', '.txt'}
WINDOW_SIZE = 5
VALID_AA = set('ACDEFGHIKLMNPQRSTVWY')

ALLOWED_AMIDE = {'Q', 'N'}
ALLOWED_REMAINDER = set('EDRKFLIVSTY')
NEGATIVE_RESIDUES = {'E', 'D'}
POSITIVE_RESIDUES = {'R', 'K'}
HYDROPHOBIC_RESIDUES = {'F', 'L', 'I', 'V'}
PHOSPHO_RESIDUES = {'S', 'T', 'Y'}

PSSM_URL_CANDIDATES = {
    'cPSSM': ['https://raw.githubusercontent.com/Marcin-Lubocki/CMAscan/main/PSSM/cPSSM.pkl'],
    'ePSSM': ['https://raw.githubusercontent.com/Marcin-Lubocki/CMAscan/main/PSSM/ePSSM.pkl'],
}

# MusiteDeep integration: planned for v1.1

IUPRED3_TARBALL_URL = 'https://raw.githubusercontent.com/Marcin-Lubocki/CMAscan/main/ExternalSoftware/iupred3.tar.gz'
IUPRED3_MODE = 'long'
IUPRED3_MIN_SEQUENCE_LENGTH = 30
IUPRED3_TIMEOUT_SEC = 180

# Default pseudo-PTM options (overwritten by form cell before run).
USE_PSEUDOACETYLATION = True
USE_PSEUDOPHOSPHORYLATION = True
PSEUDOPHOSPHO_RESIDUE = 'E'
PTM_OFF_VALUE = 'Off'

C_PSSM_MATRIX: Any | None = None
E_PSSM_MATRIX: Any | None = None


Core libraries and global constants are loaded to keep the run reproducible.


In [ ]:
#@title Step 3: Install BioPython
%pip install -q biopython tqdm
print('Dependencies installed: biopython, tqdm')


BioPython is installed in the current Colab runtime (run once per fresh session).


In [ ]:
#@title Step 4: Data Models and Input Validation
@dataclass(frozen=True)
class FastaRecord:
    header: str
    sequence: str

    @property
    def protein_name(self) -> str:
        return self.header.split()[0]


@dataclass(frozen=True)
class MotifCandidate:
    original_mer: str
    oriented_mer: str
    localization: int
    window_start_idx: int
    reoriented: bool


def include_ptm_simulation_columns() -> bool:
    return USE_PSEUDOACETYLATION or USE_PSEUDOPHOSPHORYLATION


def validate_ptm_options() -> None:
    if PSEUDOPHOSPHO_RESIDUE not in {'E', 'D'}:
        raise ValueError("PSEUDOPHOSPHO_RESIDUE must be 'E' or 'D'.")


# MusiteDeep integration: planned for v1.1

def print_ptm_simulation_summary() -> None:
    if not include_ptm_simulation_columns():
        print('Pseudo-PTM simulation for PSSM/IUPred3: OFF')
        return
    acetyl_text = 'ON (K->Q)' if USE_PSEUDOACETYLATION else f'OFF ({PTM_OFF_VALUE})'
    phospho_text = (
        f'ON (S/T/Y->{PSEUDOPHOSPHO_RESIDUE})'
        if USE_PSEUDOPHOSPHORYLATION
        else f'OFF ({PTM_OFF_VALUE})'
    )
    print(f'Pseudo-PTM simulation for PSSM/IUPred3: acetyl={acetyl_text}; phospho={phospho_text}')


def validate_input_file(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(f'Input file not found: {path}')
    if path.suffix.lower() not in ALLOWED_EXTENSIONS:
        allowed = ', '.join(sorted(ALLOWED_EXTENSIONS))
        raise ValueError(f'Invalid input format: {path.suffix}. Allowed: {allowed}')


def parse_multifasta(path: Path) -> List[FastaRecord]:
    records: List[FastaRecord] = []
    current_header: str | None = None
    current_seq_chunks: List[str] = []

    with path.open('r', encoding='utf-8') as handle:
        for raw_line in handle:
            line = raw_line.strip()
            if not line:
                continue
            if line.startswith('>'):
                if current_header is not None:
                    records.append(FastaRecord(current_header, ''.join(current_seq_chunks).upper()))
                current_header = line[1:].strip()
                current_seq_chunks = []
            else:
                current_seq_chunks.append(line)

    if current_header is not None:
        records.append(FastaRecord(current_header, ''.join(current_seq_chunks).upper()))

    if not records:
        raise ValueError('No FASTA records detected. Check file structure.')

    return records


def sanity_check_records(records: Sequence[FastaRecord], allow_x: bool = True) -> None:
    invalid_entries: List[str] = []
    for rec in records:
        if not rec.sequence:
            invalid_entries.append(f'{rec.protein_name}: empty sequence')
            continue
        for aa in rec.sequence:
            if aa in VALID_AA:
                continue
            if allow_x and aa == 'X':
                continue
            invalid_entries.append(f'{rec.protein_name}: invalid residue {aa}')
            break
    if invalid_entries:
        raise ValueError('Sanity check failed: ' + '; '.join(invalid_entries[:5]))

FASTA parsing and sanity checks validate protein sequences before scoring.


In [ ]:
#@title Step 5: Sliding Window and Orientation Rules
def iter_windows(sequence: str, k: int = WINDOW_SIZE) -> Iterator[tuple[int, str]]:
    if len(sequence) < k:
        return
    for start_idx in range(0, len(sequence) - k + 1):
        yield start_idx, sequence[start_idx:start_idx + k]


def reorient(window: str) -> str:
    return window[::-1]


def classify_window(window: str, start_idx: int) -> List[MotifCandidate]:
    candidates: List[MotifCandidate] = []
    amide_positions = [i for i, aa in enumerate(window) if aa in ALLOWED_AMIDE]
    amide_count = len(amide_positions)

    if amide_count == 1:
        amide_pos = amide_positions[0]
        if amide_pos == 0:
            candidates.append(MotifCandidate(window, window, start_idx + 1, start_idx, False))
        elif amide_pos == WINDOW_SIZE - 1:
            candidates.append(MotifCandidate(window, reorient(window), start_idx + 1, start_idx, True))
        return candidates

    if amide_count == 0:
        first_is_k = window[0] == 'K'
        last_is_k = window[-1] == 'K'

        if first_is_k and last_is_k:
            candidates.append(MotifCandidate(window, window, start_idx + 1, start_idx, False))
            candidates.append(MotifCandidate(window, reorient(window), start_idx + 1, start_idx, True))
            return candidates

        if first_is_k:
            candidates.append(MotifCandidate(window, window, start_idx + 1, start_idx, False))
            return candidates

        if last_is_k:
            candidates.append(MotifCandidate(window, reorient(window), start_idx + 1, start_idx, True))
            return candidates

    return candidates


A 5-residue sliding window with orientation rules generates motif candidates.


In [ ]:
#@title Step 6: Strict Motif Type Classification
def summarize_remainder(remainder: str) -> Dict[str, int | bool]:
    return {
        'all_allowed': all(aa in ALLOWED_REMAINDER for aa in remainder),
        'negative_count': sum(1 for aa in remainder if aa in NEGATIVE_RESIDUES),
        'positive_count': sum(1 for aa in remainder if aa in POSITIVE_RESIDUES),
        'hydrophobic_count': sum(1 for aa in remainder if aa in HYDROPHOBIC_RESIDUES),
        'phospho_count': sum(1 for aa in remainder if aa in PHOSPHO_RESIDUES),
    }


def classify_motif_type(oriented_mer: str) -> str | None:
    amide_positions = [i for i, aa in enumerate(oriented_mer) if aa in ALLOWED_AMIDE]
    amide_count = len(amide_positions)

    if amide_count == 0:
        if oriented_mer[0] != 'K':
            return None
        stats = summarize_remainder(oriented_mer[1:])
        if not stats['all_allowed']:
            return None
        if stats['negative_count'] != 1:
            return None
        if stats['positive_count'] < 1:
            return None
        if stats['hydrophobic_count'] < 1:
            return None
        if stats['phospho_count'] != 0:
            return None
        return 'acetyl'

    if amide_count == 1 and amide_positions[0] == 0:
        stats = summarize_remainder(oriented_mer[1:])
        if not stats['all_allowed']:
            return None

        is_canonical = (
            stats['negative_count'] == 1
            and stats['positive_count'] >= 1
            and stats['hydrophobic_count'] >= 1
            and stats['phospho_count'] == 0
        )
        if is_canonical:
            return 'canonical'

        is_phospho = (
            stats['phospho_count'] == 1
            and stats['positive_count'] >= 1
            and stats['hydrophobic_count'] >= 1
            and stats['negative_count'] == 0
        )
        if is_phospho:
            return 'phospho'

    return None

Motif candidates are classified as canonical, phospho, or acetyl using strict composition rules.


In [ ]:
#@title Step 7: Load cPSSM and ePSSM
def load_pickle_from_url(url: str) -> Any:
    with urllib.request.urlopen(url, timeout=30) as response:
        return pickle.loads(response.read())


def load_pssm_matrix(url_candidates: Sequence[str], matrix_name: str) -> tuple[Any, str]:
    errors: List[str] = []
    for url in url_candidates:
        try:
            return load_pickle_from_url(url), url
        except Exception as exc:  # noqa: BLE001
            errors.append(f'{url} -> {exc}')
    raise RuntimeError('Failed to download ' + matrix_name + '. Tried: ' + '; '.join(errors))


def ensure_pssm_matrices_loaded() -> None:
    global C_PSSM_MATRIX, E_PSSM_MATRIX

    try:
        from Bio import motifs as _bio_motifs  # noqa: F401
    except Exception as exc:  # noqa: BLE001
        raise RuntimeError('BioPython is required. Run the installation cell first.') from exc

    if C_PSSM_MATRIX is None:
        C_PSSM_MATRIX, c_url = load_pssm_matrix(PSSM_URL_CANDIDATES['cPSSM'], 'cPSSM')
        print(f'Loaded cPSSM from {c_url}')

    if E_PSSM_MATRIX is None:
        E_PSSM_MATRIX, e_url = load_pssm_matrix(PSSM_URL_CANDIDATES['ePSSM'], 'ePSSM')
        print(f'Loaded ePSSM from {e_url}')

---

IUPred3 disorder scores are computed using the IUPred3 tool
(https://iupred3.elte.hu/), downloaded automatically at runtime.

**If you use CMAscan output in a publication, please cite the IUPred3 authors:**

- Erdős G, Pajkos M, Dosztányi Z. IUPred3: prediction of protein disorder
  enhanced with unambiguous experimental annotation and visualization of
  evolutionary conservation. *Nucleic Acids Research* 2021;49(W1):W297–W303.
  https://academic.oup.com/nar/article/49/W1/W297/6287841

- Mészáros B, Erdős G, Dosztányi Z. IUPred2A: context-dependent prediction
  of protein disorder as a function of redox state and protein binding.
  *Nucleic Acids Research* 2018;46(W1):W329–W337.
  https://doi.org/10.1093/nar/gky384

- Erdős G, Dosztányi Z. Analyzing Protein Disorder with IUPred2A.
  *Current Protocols in Bioinformatics* 2020;70(1):e99.
  https://doi.org/10.1002/cpbi.99

---


In [ ]:
#@title Step 8: IUPred3 Disorder Scoring
def default_iupred_install_root() -> Path:
    return Path('/content/ExternalSoftware')


def safe_extract_tar_gz(tar_path: Path, target_dir: Path) -> None:
    target_dir_resolved = target_dir.resolve()
    with tarfile.open(tar_path, 'r:gz') as tar:
        for member in tar.getmembers():
            member_path = (target_dir / member.name).resolve()
            if not str(member_path).startswith(str(target_dir_resolved)):
                raise RuntimeError(f'Unsafe tar member path: {member.name}')
        tar.extractall(target_dir)


def ensure_iupred3_script() -> Path:
    install_root = default_iupred_install_root()
    install_root.mkdir(parents=True, exist_ok=True)
    script = install_root / 'iupred3' / 'iupred3.py'
    if script.exists():
        return script

    tarball_path = install_root / 'iupred3.tar.gz'
    print('Downloading IUPred3 package:', IUPRED3_TARBALL_URL)
    urllib.request.urlretrieve(IUPRED3_TARBALL_URL, tarball_path)
    safe_extract_tar_gz(tarball_path, install_root)

    if not script.exists():
        raise RuntimeError(f'IUPred3 script not found after extraction: {script}')
    return script


def parse_iupred3_output(raw_output: str) -> Dict[int, float]:
    scores: Dict[int, float] = {}
    for line in raw_output.splitlines():
        line = line.strip()
        if not line or line.startswith('#') or line.startswith('POS'):
            continue
        parts = line.split()
        if len(parts) < 3:
            continue
        try:
            pos = int(parts[0])
            val = float(parts[2])
        except ValueError:
            continue
        scores[pos] = val
    return scores


def run_iupred3_long(sequence: str, iupred_script: Path) -> Dict[int, float]:
    # Long mode should be used only for proteins potentially having long disordered regions.
    if len(sequence) < IUPRED3_MIN_SEQUENCE_LENGTH:
        return {}

    with tempfile.NamedTemporaryFile(mode='w', delete=False, suffix='.fasta') as tmp:
        tmp.write(f'>query\n{sequence}\n')
        tmp_path = Path(tmp.name)

    command = ['python3', str(iupred_script), str(tmp_path), IUPRED3_MODE]
    try:
        result = subprocess.run(
            command,
            capture_output=True,
            text=True,
            check=True,
            timeout=IUPRED3_TIMEOUT_SEC,
            cwd=str(iupred_script.parent),
        )
    except subprocess.CalledProcessError as exc:
        stderr = (exc.stderr or '').strip()
        stdout = (exc.stdout or '').strip()
        raise RuntimeError(f'IUPred3 failed: {stderr or stdout or exc}') from exc
    except subprocess.TimeoutExpired as exc:
        raise RuntimeError(f'IUPred3 timed out after {IUPRED3_TIMEOUT_SEC}s') from exc
    finally:
        tmp_path.unlink(missing_ok=True)

    return parse_iupred3_output(result.stdout)


def build_iupred3_cache(records: Sequence[FastaRecord], iupred_script: Path) -> Dict[str, Dict[int, float]]:
    cache: Dict[str, Dict[int, float]] = {}
    print(f'IUPred3 mode: {IUPRED3_MODE}')
    print(f'IUPred3 script: {iupred_script}')

    for rec in records:
        if len(rec.sequence) < IUPRED3_MIN_SEQUENCE_LENGTH:
            cache[rec.protein_name] = {}
            continue

        try:
            cache[rec.protein_name] = run_iupred3_long(rec.sequence, iupred_script)
        except Exception as exc:  # noqa: BLE001
            print(f'Warning: IUPred3 failed for {rec.protein_name}: {exc}')
            cache[rec.protein_name] = {}

    return cache


def compute_iupred3_window_mean(iupred_scores: Dict[int, float], window_start_idx: int, window_size: int = WINDOW_SIZE) -> float | None:
    positions = range(window_start_idx + 1, window_start_idx + window_size + 1)
    values = [iupred_scores[pos] for pos in positions if pos in iupred_scores]
    if len(values) != window_size:
        return None
    return float(sum(values) / len(values))

IUPred3 long-mode helpers are prepared to estimate local disorder context.


In [ ]:
#@title Step 9: PSSM + PTM Mimetic Helpers
def oriented_index_to_sequence_position(candidate: MotifCandidate, oriented_index: int) -> int:
    if candidate.reoriented:
        original_offset = (WINDOW_SIZE - 1) - oriented_index
    else:
        original_offset = oriented_index
    return candidate.window_start_idx + original_offset + 1

def get_ptm_target_position(sequence: str, candidate: MotifCandidate, motif_type: str) -> tuple[int, str] | None:
    if motif_type == 'phospho':
        sty_indices = [i for i, aa in enumerate(candidate.oriented_mer) if aa in PHOSPHO_RESIDUES]
        if len(sty_indices) != 1:
            return None
        pos_1b = oriented_index_to_sequence_position(candidate, sty_indices[0])
        if not (1 <= pos_1b <= len(sequence)):
            return None
        residue = sequence[pos_1b - 1]
        if residue not in PHOSPHO_RESIDUES:
            return None
        return pos_1b, residue

    if motif_type == 'acetyl':
        pos_1b = oriented_index_to_sequence_position(candidate, 0)
        if not (1 <= pos_1b <= len(sequence)):
            return None
        residue = sequence[pos_1b - 1]
        if residue != 'K':
            return None
        return pos_1b, residue

    return None

def get_pssm_position_score(pssm_matrix: Any, aa: str, pos: int) -> float:
    try:
        return float(pssm_matrix[aa][pos])
    except Exception:  # noqa: BLE001
        pass
    if hasattr(pssm_matrix, 'pssm'):
        return get_pssm_position_score(pssm_matrix.pssm, aa, pos)
    if isinstance(pssm_matrix, dict) and aa in pssm_matrix:
        return float(pssm_matrix[aa][pos])
    raise KeyError(f"Amino acid {aa} missing in PSSM")


def compute_pssm_score(mer: str, pssm_matrix: Any) -> float:
    motif = mer.replace('U', 'C')
    score = 0.0
    for i, aa in enumerate(motif):
        try:
            score += get_pssm_position_score(pssm_matrix, aa, i)
        except KeyError:
            score += 0.0
    return float(score)


def replace_residue_at_position(sequence: str, position_1b: int, new_residue: str) -> str:
    idx = position_1b - 1
    return sequence[:idx] + new_residue + sequence[idx + 1:]


def get_phospho_oriented_index(oriented_mer: str) -> int | None:
    indices = [i for i, aa in enumerate(oriented_mer) if aa in PHOSPHO_RESIDUES]
    if len(indices) != 1:
        return None
    return indices[0]


def build_ptm_mimetic_mers(candidate: MotifCandidate, motif_type: str) -> tuple[str, str] | None:
    if motif_type == 'acetyl':
        replacement = 'Q'
        target_index = 0
    elif motif_type == 'phospho':
        target_index = get_phospho_oriented_index(candidate.oriented_mer)
        if target_index is None:
            return None
        replacement = PSEUDOPHOSPHO_RESIDUE
    else:
        return None

    oriented_chars = list(candidate.oriented_mer)
    oriented_chars[target_index] = replacement
    oriented_mimetic = ''.join(oriented_chars)
    original_mimetic = oriented_mimetic[::-1] if candidate.reoriented else oriented_mimetic
    return original_mimetic, oriented_mimetic


def compute_iupred3_ptm_window_mean(
    sequence: str,
    window_start_idx: int,
    target_pos_1b: int,
    replacement_residue: str,
    iupred_script: Path,
    cache: Dict[tuple[str, int, str], Dict[int, float]],
) -> float | None:
    if len(sequence) < IUPRED3_MIN_SEQUENCE_LENGTH:
        return None

    cache_key = (sequence, target_pos_1b, replacement_residue)
    if cache_key not in cache:
        mutated_sequence = replace_residue_at_position(sequence, target_pos_1b, replacement_residue)
        try:
            cache[cache_key] = run_iupred3_long(mutated_sequence, iupred_script)
        except Exception as exc:  # noqa: BLE001
            print(f'Warning: IUPred3 PTM rerun failed (pos={target_pos_1b}, replace={replacement_residue}): {exc}')
            cache[cache_key] = {}

    return compute_iupred3_window_mean(cache[cache_key], window_start_idx, WINDOW_SIZE)

Helper functions define pseudo-PTM mimetics and rescoring logic.


In [ ]:
#@title Step 10: Build Pipeline and Export CSV
from tqdm.notebook import tqdm
def find_all_motif_hits(
    records: Sequence[FastaRecord],
    iupred3_cache: Dict[str, Dict[int, float]],
    iupred_script: Path,
) -> List[Dict[str, str | int | float]]:
    rows: List[Dict[str, str | int | float]] = []
    use_ptm_columns = include_ptm_simulation_columns()
    ptm_iupred_cache: Dict[tuple[str, int, str], Dict[int, float]] = {}

    for rec in tqdm(records, desc='Scanning proteins', unit='protein'):
        record_hits = 0
        rec_iupred_scores = iupred3_cache.get(rec.protein_name, {})

        for start_idx, window in iter_windows(rec.sequence, WINDOW_SIZE):
            candidates = classify_window(window, start_idx)
            for cand in candidates:
                motif_type = classify_motif_type(cand.oriented_mer)
                if motif_type is None:
                    continue

                target = get_ptm_target_position(rec.sequence, cand, motif_type)
                # MusiteDeep integration: planned for v1.1

                iupred_value = compute_iupred3_window_mean(rec_iupred_scores, cand.window_start_idx, WINDOW_SIZE)

                mer_ptm_mimetic: str | float = 'NA'
                c_pssm_ptm: str | float = 'NA'
                e_pssm_ptm: str | float = 'NA'
                iupred3_ptm: str | float = 'NA'

                if use_ptm_columns:
                    if motif_type == 'canonical':
                        mer_ptm_mimetic = 'NA'
                        c_pssm_ptm = 'NA'
                        e_pssm_ptm = 'NA'
                        iupred3_ptm = 'NA'
                    elif motif_type == 'acetyl' and not USE_PSEUDOACETYLATION:
                        mer_ptm_mimetic = PTM_OFF_VALUE
                        c_pssm_ptm = PTM_OFF_VALUE
                        e_pssm_ptm = PTM_OFF_VALUE
                        iupred3_ptm = PTM_OFF_VALUE
                    elif motif_type == 'phospho' and not USE_PSEUDOPHOSPHORYLATION:
                        mer_ptm_mimetic = PTM_OFF_VALUE
                        c_pssm_ptm = PTM_OFF_VALUE
                        e_pssm_ptm = PTM_OFF_VALUE
                        iupred3_ptm = PTM_OFF_VALUE
                    else:
                        mimetic = build_ptm_mimetic_mers(cand, motif_type)
                        if mimetic is not None and target is not None:
                            mer_ptm_mimetic, oriented_mimetic_mer = mimetic
                            c_pssm_ptm = round(compute_pssm_score(oriented_mimetic_mer, C_PSSM_MATRIX), 2)
                            e_pssm_ptm = round(compute_pssm_score(oriented_mimetic_mer, E_PSSM_MATRIX), 2)

                            ptm_pos_1b, _ = target
                            replacement_residue = 'Q' if motif_type == 'acetyl' else PSEUDOPHOSPHO_RESIDUE
                            ptm_iupred_value = compute_iupred3_ptm_window_mean(
                                rec.sequence,
                                cand.window_start_idx,
                                ptm_pos_1b,
                                replacement_residue,
                                iupred_script,
                                ptm_iupred_cache,
                            )
                            iupred3_ptm = round(ptm_iupred_value, 2) if ptm_iupred_value is not None else 'NA'

                row = {
                    'protein_name': rec.protein_name,
                    'mer': cand.original_mer,
                    'type': motif_type,
                    'localization': int(cand.localization),
                    'cPSSM': round(compute_pssm_score(cand.oriented_mer, C_PSSM_MATRIX), 2),
                    'ePSSM': round(compute_pssm_score(cand.oriented_mer, E_PSSM_MATRIX), 2),
                    'IUPred3': round(iupred_value, 2) if iupred_value is not None else 'NA',
                    # MusiteDeep integration: planned for v1.1
                }
                if use_ptm_columns:
                    row['mer_PTM_mimetic'] = mer_ptm_mimetic
                    row['cPSSM_PTM'] = c_pssm_ptm
                    row['ePSSM_PTM'] = e_pssm_ptm
                    row['IUPred3_PTM'] = iupred3_ptm

                rows.append(row)
                record_hits += 1

        if record_hits == 0:
            empty_row: Dict[str, str | int | float] = {
                'protein_name': rec.protein_name,
                'mer': 'NA',
                'type': 'No motifs found',
                'localization': 'NA',
                'cPSSM': 'NA',
                'ePSSM': 'NA',
                'IUPred3': 'NA',
                # MusiteDeep integration: planned for v1.1
            }
            if use_ptm_columns:
                empty_row['mer_PTM_mimetic'] = 'NA'
                empty_row['cPSSM_PTM'] = 'NA'
                empty_row['ePSSM_PTM'] = 'NA'
                empty_row['IUPred3_PTM'] = 'NA'
            rows.append(empty_row)

    return rows


def write_csv(rows: Sequence[Dict[str, str | int | float]], output_path: Path) -> None:
    columns = ['protein_name', 'mer', 'type', 'localization', 'cPSSM', 'ePSSM', 'IUPred3']
    if include_ptm_simulation_columns():
        columns.extend(['mer_PTM_mimetic', 'cPSSM_PTM', 'ePSSM_PTM', 'IUPred3_PTM'])
    # MusiteDeep integration: planned for v1.1

    with output_path.open('w', encoding='utf-8', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=columns, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(rows)


def run_pipeline(input_path: str, output_csv: str = 'motif_hits_scored.csv') -> Path:
    validate_ptm_options()
    print_ptm_simulation_summary()

    in_path = Path(input_path)
    out_path = Path(output_csv)

    validate_input_file(in_path)
    records = parse_multifasta(in_path)
    sanity_check_records(records, allow_x=True)

    ensure_pssm_matrices_loaded()
    iupred_script = ensure_iupred3_script()
    iupred3_cache = build_iupred3_cache(records, iupred_script)
    # MusiteDeep integration: planned for v1.1

    rows = find_all_motif_hits(records, iupred3_cache, iupred_script)
    write_csv(rows, out_path)

    print()
    print('--- CMAscan run complete ---')
    print(f'  Proteins processed : {len(records)}')
    motif_rows = [r for r in rows if r.get("type") != "No motifs found"]
    print(f'  Motif hits found   : {len(motif_rows)}')
    print(f'  Rows in CSV        : {len(rows)}')
    print(f'  Output file        : {out_path.resolve()}')
    print()
    print('If results look incomplete or an error occurred, click')
    print('Runtime -> Disconnect and delete runtime, then Runtime -> Run all.')

    return out_path

The end-to-end pipeline function is assembled and ready to execute.


In [ ]:
#@title Step 11: Run Analysis
result_path = run_pipeline(input_path=INPUT_FILE_PATH, output_csv=OUTPUT_FILE_PATH)

The full analysis is executed on the selected FASTA input and exported as CSV.


In [ ]:
#@title Step 12: Preview and Download CSV
df = pd.read_csv(result_path)
display(df.head(30))
print(f'Total rows: {len(df)}')
print('Type distribution:')
print(df['type'].value_counts(dropna=False))
print()
print('Output column legend:')
legend = {
    'protein_name': 'Identifier from FASTA header',
    'mer': '5-residue motif candidate in original sequence orientation',
    'type': 'canonical | phospho | acetyl | "No motifs found"',
    'localization': '1-based start position of the motif in the protein sequence',
    'cPSSM': 'Score from canonical PSSM (canonical CMA motifs)',
    'ePSSM': 'Score from extended PSSM (canonical + atypical CMA motifs)',
    'IUPred3': 'Mean disorder score over the 5 motif residues (0 = ordered, 1 = disordered)',
    'mer_PTM_mimetic': 'Motif sequence after simulated PTM (K -> Q for acetyl; S/T/Y -> E or D for phospho)',
    'cPSSM_PTM': 'cPSSM score recomputed for the PTM mimetic motif',
    'ePSSM_PTM': 'ePSSM score recomputed for the PTM mimetic motif',
    'IUPred3_PTM': 'Disorder score recomputed on sequence with PTM mimetic substitution',
}
for col, desc in legend.items():
    if col in df.columns:
        print(f'  {col}: {desc}')
files.download(str(result_path))